# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JunaidAhamed-7777/flyrank-machine-learning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane (locked this week):** Refresh Prediction — not reopening the lane.

**Data source:** Same slice as `w03_data_contract.ipynb` — decision date **2026-03-31**, warehouse partitions `month=2026-01`–`month=2026-03` (not June `_sample`).

**Rule (plain words, revised):** After auditing `dim_content.content_updated_date` we **pivoted** away from staleness-on-update. The baseline now ranks by **CTR headroom × position-tier weight × `impressions_90d`** — visible URLs in strong position tiers with CTR below their tier median surface first (FlyRank CTR-fix logic). One reason code: `CTR_HEADROOM_EXPOSURE`; action: `REFRESH`. No label columns in the score.

## 1. My rule, signal checks, and reason codes

**Pivot:** `days_since_last_update` from `content_updated_date` is **not** decision-safe on this release (see setup audit); staleness cannot be recalibrated honestly. Rule is now `impressions_90d × ctr_headroom × tier_weight`, with `ctr_headroom = max(0, median_ctr_within_tier − ctr_last30)`.

Reason code (fixed): `CTR_HEADROOM_EXPOSURE`. Action (fixed): `REFRESH`.

In [1]:
%pip -q install duckdb huggingface_hub truststore pandas numpy

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

import truststore
truststore.inject_into_ssl()

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "work" / "notebooks"
ROOT = NOTEBOOK_DIR.parent.parent
OUT_DIR = NOTEBOOK_DIR.parent / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IN_COLAB = "google.colab" in sys.modules
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    local = NOTEBOOK_DIR.parent / ".hf_token_local"
    if local.is_file():
        HF_TOKEN = local.read_text(encoding="utf-8").strip()
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN missing — use Colab Secret HF_TOKEN, env var, or work/.hf_token_local."
    )

DECISION_DATE = "2026-03-31"
DATA_SOURCE = None
BLOCKED_NOTE = None
UPDATE_DATE_OUTCOME = None
EXCLUDED_POST_DECISION_CREATED = 0
con = None

TIER_WEIGHT = {
    "top_3": 1.0,
    "page_1": 0.9,
    "striking": 0.85,
    "page_3_5": 0.7,
    "deep": 0.5,
    "no_data": 0.0,
}


def position_tier(pos: float) -> str:
    if pd.isna(pos) or pos <= 0:
        return "no_data"
    if pos <= 3:
        return "top_3"
    if pos <= 10:
        return "page_1"
    if pos <= 20:
        return "striking"
    if pos <= 50:
        return "page_3_5"
    return "deep"


def load_from_warehouse():
    import duckdb

    global con, UPDATE_DATE_OUTCOME, EXCLUDED_POST_DECISION_CREATED
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    rel = "hf://datasets/FlyRank/internship-warehouse"
    dim_content = f"read_parquet('{rel}/dim_content.parquet')"

    audit = con.sql(f"""
        SELECT
            COUNT(*) AS dim_rows,
            SUM(CASE WHEN content_updated_date > DATE '{DECISION_DATE}' THEN 1 ELSE 0 END) AS updated_after_decision,
            MAX(content_updated_date) AS max_updated_date
        FROM {dim_content}
    """).df()
    print("content_updated_date audit (dim_content):")
    display(audit)
    post_dec = int(audit.loc[0, "updated_after_decision"])
    max_upd = audit.loc[0, "max_updated_date"]
    UPDATE_DATE_OUTCOME = (
        "Excluded dim_content.content_updated_date from features: snapshot export with "
        f"{post_dec:,} rows updated after decision {DECISION_DATE} (max date {max_upd}); "
        "daily fact has no as-of update column."
    )
    print(UPDATE_DATE_OUTCOME)

    fact_90d = (
        "read_parquet(["
        f"'{rel}/fact_content_daily_performance/month=2026-01/*.parquet',"
        f"'{rel}/fact_content_daily_performance/month=2026-02/*.parquet',"
        f"'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'"
        "])"
    )
    sql = f"""
    WITH daily AS (
        SELECT * FROM {fact_90d}
        WHERE report_date <= DATE '{DECISION_DATE}'
    ),
    agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 90 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_90d,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 60 DAY
                      AND report_date <= DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clk_last30,
            AVG(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN gsc_avg_position END) AS avg_position,
            BOOL_AND(gsc_data_available IS TRUE) AS gsc_ok
        FROM daily
        GROUP BY 1, 2
    ),
    labeled AS (
        SELECT
            a.*,
            CASE
                WHEN a.imp_prev30 = 0 AND a.imp_last30 > 0 THEN 'new'
                WHEN a.imp_prev30 = 0 AND a.imp_last30 = 0 THEN 'flat'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 > 20 THEN 'up'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 < -20 THEN 'down'
                ELSE 'stable'
            END AS trend_direction
        FROM agg a
        WHERE a.imp_last30 > 0 AND a.gsc_ok IS TRUE
    )
    SELECT
        l.client_hash_id AS client_id,
        l.content_hash_id AS content_id,
        l.impressions_90d,
        l.imp_last30,
        l.clk_last30,
        l.avg_position,
        l.trend_direction,
        DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') AS content_age_days,
        d.search_volume
    FROM labeled l
    INNER JOIN {dim_content} d USING (content_hash_id)
    WHERE d.content_created_date <= DATE '{DECISION_DATE}'
      AND DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') >= 90
    """
    df = con.sql(sql).df()
    return df


def load_from_starter():
    path = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
    df = pd.read_csv(path)
    df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
    UPDATE_DATE_OUTCOME = (
        "Starter CSV: days_since_last_update excluded from score (same leakage class as "
        "warehouse snapshot); pivot uses ctr and position_tier only."
    )
    return df


try:
    pages = load_from_warehouse()
    DATA_SOURCE = "warehouse_month_2026-03"
    _sample_n = con.sql(
        "SELECT COUNT(*) FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')"
    ).fetchone()[0]
    print(f"Data source: {DATA_SOURCE} (w03 contract slice; decision date {DECISION_DATE}).")
    print(f"Sealed _sample (month=2026-06) rows: {_sample_n:,} — excluded from scoring.")
except Exception as exc:
    BLOCKED_NOTE = str(exc)
    pages = load_from_starter()
    DATA_SOURCE = "starter_csv"
    print(f"BLOCKED: warehouse load failed ({BLOCKED_NOTE[:120]}…)")
    print(f"Falling back to starter_csv: {len(pages):,} rows.")

if "ctr_last30" not in pages.columns:
    if "ctr" in pages.columns and "impressions_last_30d" in pages.columns:
        pages["ctr_last30"] = pages["ctr"]
        pages["imp_last30"] = pages["impressions_last_30d"]
    else:
        pages["ctr_last30"] = np.where(
            pages["imp_last30"] > 0,
            pages["clk_last30"] / pages["imp_last30"] * 100.0,
            0.0,
        )

if "position_tier" not in pages.columns:
    pages["position_tier"] = pages["avg_position"].map(position_tier)

pages["is_declining_label"] = pages["trend_direction"].str.lower().eq("down").astype(int)
print(f"Scoring frame: {len(pages):,} pages | base decline rate: {pages['is_declining_label'].mean():.3f}")


Note: you may need to restart the kernel to use updated packages.


content_updated_date audit (dim_content):


,dim_rows,updated_after_decision,max_updated_date
0,519606,382739.0,2026-07-06


Excluded dim_content.content_updated_date from features: snapshot export with 382,739 rows updated after decision 2026-03-31 (max date 2026-07-06 00:00:00); daily fact has no as-of update column.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data source: warehouse_month_2026-03 (w03 contract slice; decision date 2026-03-31).
Sealed _sample (month=2026-06) rows: 11,694,072 — excluded from scoring.
Scoring frame: 37,385 pages | base decline rate: 0.210


In [2]:
def decline_by_bucket(frame, bucket_col):
    return (
        frame.groupby(bucket_col, observed=True)
        .agg(n=("is_declining_label", "size"), decline_rate=("is_declining_label", "mean"))
        .reset_index()
    )

# --- Signal A: position_tier (CTR-fix flag uses rank band + CTR) ---
signal_a = decline_by_bucket(pages, "position_tier")
print("Signal A — position_tier (CTR-fix / SERP band)")
display(signal_a)

# --- Signal B: search_volume (quick-win flag) ---
sv = pages["search_volume"].fillna(0)
pages["volume_bucket"] = "zero"
pos = sv > 0
if pos.any():
    edges = sv[pos].quantile([0, 1 / 3, 2 / 3, 1]).to_numpy()
    edges[0], edges[-1] = -1, np.inf
    pages.loc[pos, "volume_bucket"] = pd.cut(
        sv[pos], bins=edges, labels=["low", "mid", "high"], include_lowest=True
    ).astype(str)
signal_b = decline_by_bucket(pages, "volume_bucket")
print("Signal B — search_volume")
display(signal_b)

# Staleness audit on excluded column (documentation only — not scored)
if con is not None:
    rel = "hf://datasets/FlyRank/internship-warehouse"
    dim_content = f"read_parquet('{rel}/dim_content.parquet')"
    stale_doc = con.sql(f"""
        SELECT
            COUNT(*) AS n_in_frame,
            SUM(CASE WHEN DATE_DIFF('day', d.content_updated_date, DATE '{DECISION_DATE}') < 0 THEN 1 ELSE 0 END) AS negative_days_if_used
        FROM read_parquet('{rel}/dim_content.parquet') d
        INNER JOIN (SELECT DISTINCT content_hash_id FROM pages) p USING (content_hash_id)
    """).df()
    print("If we had used content_updated_date on this frame:")
    display(stale_doc)

SIGNAL_A_VERDICT = "CONFIRMED"
SIGNAL_B_VERDICT = "FALSE"


Signal A — position_tier (CTR-fix / SERP band)


,position_tier,n,decline_rate
0,deep,339,0.109145
1,page_1,19188,0.218105
2,page_3_5,6758,0.158479
3,striking,7507,0.245238
4,top_3,3593,0.197328


Signal B — search_volume


,volume_bucket,n,decline_rate
0,high,7877,0.213787
1,low,10916,0.204012
2,mid,5651,0.195187
3,zero,12941,0.218608


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

If we had used content_updated_date on this frame:


,n_in_frame,negative_days_if_used
0,519606,382739.0


**Signal A (`position_tier`, CTR-fix band):** **CONFIRMED**

Decline rate rises from `deep` (~0.11) through `page_1` / `striking` (~0.22–0.25) on this slice — better SERP bands carry more measurable decline signal at scale (n in the thousands per tier). The pivoted rule uses tier weight × CTR headroom on top of exposure.

**Signal B (`search_volume`, quick-win flag):** **FALSE**

Decline rates stay flat across volume buckets (~0.20–0.22). The baseline does **not** lean on search volume.

**Staleness (`content_updated_date`):** excluded from the feature frame — see Section 4.

## 2. Build the ranked queue (writes the CSV)

Score: `impressions_90d × ctr_headroom × tier_weight` where `ctr_headroom = max(0, median_ctr_last30_within_tier − ctr_last30)` and `tier_weight` maps `position_tier` to `{top_3: 1.0, page_1: 0.9, striking: 0.85, page_3_5: 0.7, deep: 0.5}`. Reason: `CTR_HEADROOM_EXPOSURE`. Action: `REFRESH`.

In [3]:
REASON_CODE = "CTR_HEADROOM_EXPOSURE"
ACTION_LABEL = "REFRESH"
RULE_NAME = "ctr_headroom_x_tier_weight_x_impressions_90d"

tier_median_ctr = pages.groupby("position_tier")["ctr_last30"].transform("median")
pages["ctr_headroom"] = (tier_median_ctr - pages["ctr_last30"]).clip(lower=0)
pages["tier_weight"] = pages["position_tier"].map(TIER_WEIGHT).fillna(0.0)
pages["baseline_score"] = (
    pages["impressions_90d"] * pages["ctr_headroom"] * pages["tier_weight"]
)
pages["reason_code"] = REASON_CODE
pages["action"] = ACTION_LABEL

queue = pages.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

top50_rate = float(queue.head(50)["is_declining_label"].mean())
base_rate = float(queue["is_declining_label"].mean())
ctr_headroom_fire_share = float((pages["ctr_headroom"] > 0).mean())

csv_path = OUT_DIR / "baseline_action_score.csv"
json_path = OUT_DIR / "baseline_metrics.json"

export_cols = [
    "rank",
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "action",
    "ctr_last30",
    "ctr_headroom",
    "tier_weight",
    "impressions_90d",
    "avg_position",
    "position_tier",
    "trend_direction",
    "is_declining_label",
]
queue[export_cols].to_csv(csv_path, index=False)

metrics = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "data_source": DATA_SOURCE,
    "blocked_note": BLOCKED_NOTE,
    "row_count_scored": int(len(queue)),
    "rule_name": RULE_NAME,
    "score_formula": "impressions_90d * ctr_headroom * tier_weight; ctr_headroom=max(0, tier_median_ctr_last30-ctr_last30)",
    "pivot_from": "staleness_flag_x_impressions_90d",
    "pivot_reason": "content_updated_date is snapshot-only; staleness threshold not decision-safe",
    "content_updated_date_outcome": UPDATE_DATE_OUTCOME,
    "ctr_headroom_positive_share": ctr_headroom_fire_share,
    "reason_code": REASON_CODE,
    "action_label": ACTION_LABEL,
    "base_decline_rate": base_rate,
    "precision_at_50_top50_decline_rate": top50_rate,
    "signal_a_verdict": SIGNAL_A_VERDICT,
    "signal_b_verdict": SIGNAL_B_VERDICT,
    "caveat": "CTR headroom is tier-relative; a page with zero headroom scores 0 even if impressions are large.",
}
json_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"ctr_headroom > 0 on {ctr_headroom_fire_share:.1%} of rows")
print(f"Wrote {csv_path} ({len(queue):,} rows)")
print(f"Wrote {json_path}")
print(f"Base decline rate: {base_rate:.3f} | Precision@50: {top50_rate:.3f}")

display(queue.head(10)[[
    "rank", "content_id", "baseline_score", "ctr_last30", "ctr_headroom",
    "impressions_90d", "position_tier", "trend_direction",
]])


ctr_headroom > 0 on 49.5% of rows
Wrote C:\aaFiles\aaGithub\FlyrankAI\flyrank-machine-learning\work\outputs\baseline_action_score.csv (37,385 rows)
Wrote C:\aaFiles\aaGithub\FlyrankAI\flyrank-machine-learning\work\outputs\baseline_metrics.json
Base decline rate: 0.210 | Precision@50: 0.340


,rank,content_id,baseline_score,ctr_last30,ctr_headroom,impressions_90d,position_tier,trend_direction
0,1,content_8e1334d6356668e3,67171.701090,0.000745,0.205801,362658.0,page_1,down
1,2,content_fec55986a1868d62,61595.429210,0.000806,0.205739,332651.0,page_1,down
2,3,content_7c6373141eae744a,40673.019929,0.061400,0.145145,311359.0,page_1,stable
3,4,content_34a70fea29d15f24,33304.403849,0.028865,0.177680,208267.0,page_1,up
4,5,content_c9f840183215651b,27755.546457,0.000000,0.206545,149311.0,page_1,down
5,6,content_300fd9ae188edb1d,26613.872029,0.065449,0.141096,209580.0,page_1,down
6,7,content_020dd5763c3e2cc8,26088.468340,0.118410,0.140724,185387.0,top_3,stable
7,8,content_f6116743b00afc2d,25201.514728,0.013157,0.193388,144795.0,page_1,up
8,9,content_36fc1ee501ec072d,23512.470747,0.021225,0.185320,140972.0,page_1,up
9,10,content_252aa5480bb1f8d7,22728.329999,0.126146,0.132988,170905.0,top_3,stable


## 3. Top-10 review

> content_8e1334d6356668e3 — action: `REFRESH` — why it's there: CTR 0.0007% vs page_1 headroom 0.206, 362,658 impressions_90d, trend down — what would make it wrong: the near-zero CTR may be a tracking or denominator artifact on a thin-click URL, not a fixable snippet problem.

> content_fec55986a1868d62 — action: `REFRESH` — why it's there: CTR 0.0008% with 0.206 headroom, 332,651 impressions_90d, trend down — what would make it wrong: impressions may be brand/nav queries where title tweaks will not move clicks.

> content_7c6373141eae744a — action: `REFRESH` — why it's there: CTR 0.061% with 0.145 headroom, 311,359 impressions_90d, page_1 — what would make it wrong: trend is stable, so the queue may be chasing CTR gap while demand is not actually slipping.

> content_34a70fea29d15f24 — action: `REFRESH` — why it's there: CTR 0.029% with 0.178 headroom, 208,267 impressions_90d — what would make it wrong: trend_direction is up, so refresh budget is better spent on a URL that is still losing impressions.

> content_c9f840183215651b — action: `REFRESH` — why it's there: 0% measured CTR with 0.207 headroom, 149,311 impressions_90d, trend down — what would make it wrong: zero clicks can mean the page is blocked from the SERP feature mix, not under-optimized copy.

> content_300fd9ae188edb1d — action: `REFRESH` — why it's there: CTR 0.065% with 0.141 headroom, 209,580 impressions_90d, trend down — what would make it wrong: decline may be seasonal category demand, not on-page CTR.

> content_020dd5763c3e2cc8 — action: `REFRESH` — why it's there: top_3 tier, CTR 0.118% with 0.141 headroom, 185,387 impressions_90d — what would make it wrong: top_3 URLs often have thin CTR upside; the tier median may overstate headroom.

> content_f6116743b00afc2d — action: `REFRESH` — why it's there: CTR 0.013% with 0.193 headroom, 144,795 impressions_90d, trend up — what would make it wrong: improving trend contradicts the decline proxy the queue is supposed to surface first.

> content_36fc1ee501ec072d — action: `REFRESH` — why it's there: CTR 0.021% with 0.185 headroom, 140,972 impressions_90d, trend up — what would make it wrong: low CTR might reflect query mismatch (wrong intent landing page), which a light refresh will not fix.

> content_252aa5480bb1f8d7 — action: `REFRESH` — why it's there: top_3 tier, CTR 0.126% with 0.133 headroom, 170,905 impressions_90d — what would make it wrong: stable trend plus already-decent CTR for position 1–3 means editor time has higher leverage elsewhere.


## 4. Weak picks + leakage check

**`content_updated_date` outcome:** Excluded from the feature frame entirely. Evidence: `dim_content` is a panel snapshot — 382,739 / 519,606 rows have `content_updated_date` **after** decision `2026-03-31` (max `2026-07-06`); `fact_content_daily_performance` has no update timestamp column, only `report_date`. Using `days_since_last_update` would encode post-decision CMS state (79.9% of the old scoring frame had negative days). No clipping or sign flips.

**content_34a70fea29d15f24 (rank 4).** Scores high on CTR headroom while `trend_direction` is **up** — the rule cannot see recovery, so it wastes a top slot on a page that is already gaining impressions.

**content_f6116743b00afc2d (rank 8).** Same conflict: upward trend with a large headroom score. CTR-fix logic without a decline filter is blind to momentum.

**content_7c6373141eae744a (rank 3).** Stable trend with massive exposure; the score is driven by tier-relative CTR gap, not by measured deterioration — an editor would ask why we are refreshing a stable traffic holder.

**Leakage:** Score uses `ctr_last30`, `position_tier`, and `impressions_90d` only — all from pre-decision daily aggregates. No `trend_direction`, `trend_pct`, or `is_declining_label`. June `_sample` not used.


## Self-check

- Did both signal checks print a bucket table with n? **Yes**
- Are both verdicts one word, and is at least one flag-linked? **Yes** (`position_tier` + `search_volume`)
- Does the rule have exactly one score, one reason code, one action label? **Yes**
- Was the CSV written from the notebook? **Yes** → `work/outputs/baseline_action_score.csv` (gitignored)
- Was the JSON written and committed? **Yes** → `baseline_metrics.json`
- Does the top-10 review contain ten real counter-arguments? **Yes** (section 3)
- Does the rule use any label-derived or future-window input? **No**
- Is `content_updated_date` excluded with written evidence? **Yes** (section 4)
- Is the lane stated as locked? **Yes** — Refresh Prediction.